In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
#orders_schema

item_schema = ArrayType(StructType([
    StructField("item_id", StringType()),
    StructField("name", StringType()),
    StructField("category", StringType()),
    StructField("unit_price",DecimalType(10,2)),
    StructField("subtotal", DecimalType(10,2))
])
                        
)

df_fact_orders = (spark.read.table("adb_dbxproject.`01_bronze`.orders")
            .withColumn("order_timestamp", to_timestamp(col("order_timestamp")))
            .withColumn("order_date", to_date(col("order_timestamp")))
            .withColumn("order_hour", hour(col("order_timestamp")))
            .withColumn("day_of_week", date_format(col("order_timestamp"), "EEEE"))
            .withColumn(
                "is_weekend",
                when(col("day_of_week").isin(["Saturday", "Sunday"]), True)
                .otherwise(False)
            )
            .withColumn("items_parsed", from_json(col("items"), item_schema))
            .withColumn("item_count", size(col("items_parsed")))
            ). select(
                "order_id",
                "order_timestamp",
                "order_date",
                "order_hour",
                "day_of_week",
                "is_weekend",
                "restaurant_id",
                "customer_id",
                "order_type",
                "item_count",
                col("total_amount").cast("decimal(10,2)"),
                "payment_method",
                "order_status"
                )

display(df_fact_orders)

order_id,order_timestamp,order_date,order_hour,day_of_week,is_weekend,restaurant_id,customer_id,order_type,item_count,total_amount,payment_method,order_status
ORD-20250715-104297,2025-07-15T13:59:33.174Z,2025-07-15,13,Tuesday,false,REST-AUH-001,CUST-10152,delivery,5,117.00,wallet,completed
ORD-20250715-108293,2025-07-15T11:02:16.174Z,2025-07-15,11,Tuesday,false,REST-DXB-001,CUST-10217,dine_in,5,299.00,wallet,delivered
ORD-20250715-144149,2025-07-15T21:45:12.174Z,2025-07-15,21,Tuesday,false,REST-AUH-001,CUST-10044,takeaway,5,379.00,card,completed
ORD-20250715-222203,2025-07-15T20:23:12.174Z,2025-07-15,20,Tuesday,false,REST-DXB-002,CUST-10086,delivery,3,108.00,cash,completed
ORD-20250715-262440,2025-07-15T16:20:21.174Z,2025-07-15,16,Tuesday,false,REST-DXB-002,CUST-10143,takeaway,5,480.00,wallet,delivered
ORD-20250715-274003,2025-07-15T16:23:39.174Z,2025-07-15,16,Tuesday,false,REST-SHJ-001,CUST-10274,dine_in,1,54.00,cash,completed
ORD-20250715-316377,2025-07-15T11:03:49.174Z,2025-07-15,11,Tuesday,false,REST-DXB-002,CUST-10019,delivery,4,320.00,wallet,delivered
ORD-20250715-368191,2025-07-15T18:54:24.174Z,2025-07-15,18,Tuesday,false,REST-AUH-001,CUST-10168,delivery,4,275.00,card,completed
ORD-20250715-369279,2025-07-15T13:49:21.174Z,2025-07-15,13,Tuesday,false,REST-SHJ-001,CUST-10170,takeaway,5,328.00,cash,delivered
ORD-20250715-391612,2025-07-15T14:38:47.174Z,2025-07-15,14,Tuesday,false,REST-SHJ-001,CUST-10294,dine_in,4,207.00,wallet,completed
